# Capítulo 10 — Arquitetura completa e feedback do usuário

## Missão
Montar o **assistente da Nimbus de ponta a ponta**, juntando as peças dos capítulos anteriores na ordem de uma arquitetura de produção:
1. contexto (RAG);
2. proteções de entrada e de saída;
3. roteador de modelos;
4. cache;
5. registro para monitoramento.

Depois, medir o sistema como um time de produto faria (latência, custo, bloqueios, cache) e transformar o **feedback dos usuários** em melhoria.

**No fim deste caderno você terá:** o módulo `meu_assistente/assistente.py`, com o assistente completo, um painel de métricas a partir dos registros e a comparação final com a linha de base do capítulo 4.

**Leitura essencial (opcional, livro em inglês):**
- AI Engineering Architecture: Steps 1 to 5 · PDF p. 473
- Monitoring and Observability · PDF p. 489
- User Feedback: Extracting Conversational Feedback · PDF p. 498
- Feedback Design · PDF p. 504

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap10_arquitetura_completa_e_feedback_do_usuario.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, re
sys.path.insert(0, "../comum")                 # ia.py (acesso aos modelos) e dados.py (dados da Nimbus)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_assistente/
os.makedirs("../meu_assistente", exist_ok=True)
open("../meu_assistente/__init__.py", "a").close()        # meu_assistente/ é o projeto que você constrói ao longo do curso
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
import numpy as np
import pandas as pd
from ia import *
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Modelo:", MODELO, "| modo simulado:", FAKE, "| seu pacote: meu_assistente")

## Ideia em 30 segundos
A arquitetura cresce em camadas, e cada camada só entra quando resolve um problema medido:
1. **contexto**: o modelo passa a saber o que precisa;
2. **guardrails**: barram entradas perigosas e saídas ruins;
3. **roteador**: manda cada pedido para o modelo certo, o barato quando dá;
4. **cache**: não paga duas vezes pela mesma resposta;
5. **agentes**: só quando a tarefa pede ação.

Em volta de tudo ficam **monitoramento** e **feedback**. Sem eles, você não sabe se o sistema piorou.

### 10.1 **Guardrails.** Escreva `guardrails.py` com:
- `mascarar_pii(texto)`: troca e-mail, CPF e telefone por marcadores;
- `verificar_entrada(texto)`: devolve `(ok, motivo)`; bloqueia injeção de prompt (use `detectar_injecao` do seu `defesas.py`) e mensagens vazias ou com mais de 2.000 caracteres;
- `verificar_saida(texto, segredos)`: aplica `filtrar_saida` e bloqueia respostas vazias. Devolve `(ok, texto_final)`.

_A célula grava `meu_assistente/guardrails.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/guardrails.py


### 10.2 **Teste os guardrails.** Mascare "Sou a Bia, bia@exemplo.com, (81) 99999-1234, CPF 111.222.333-44" e passe por `verificar_entrada`: uma pergunta normal, um ataque e uma mensagem vazia.

### 10.3 **Roteador.** Escreva `roteador.py` com:
- `classificar_intencao(pergunta)`: pede ao modelo barato um JSON `{"intencao": ...}` com uma de: `duvida_produto`, `preco`, `problema_conta`, `fora_do_escopo`, `falar_com_humano`. Se vier algo fora da lista, usa `duvida_produto`;
- `escolher_rota(intencao)`: devolve `{"modelo": ..., "usar_rag": bool, "responder": bool}`.

A regra: fora do escopo e falar com humano não chamam o modelo de resposta; problemas de conta vão para o modelo forte; o resto vai para o barato com RAG.

_A célula grava `meu_assistente/roteador.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/roteador.py


### 10.4 **Teste o roteador.** Classifique 5 mensagens (uma de cada tipo, escritas por você) e mostre a rota escolhida.

### 10.5 **O assistente completo.** Escreva `assistente.py` com a classe `Assistente(documentos, segredos=())`. O método `responder(pergunta, usuario="anonimo")` faz, em ordem:
1. mascara dados pessoais e roda `verificar_entrada`; se bloquear, responde com uma recusa educada;
2. consulta o **cache**;
3. classifica a intenção e escolhe a rota; nas rotas sem resposta, devolve mensagens prontas;
4. responde com o **RAG** (`BaseConhecimento.responder`) no modelo da rota;
5. passa a saída pelo `verificar_saida` e guarda no cache.

Cada chamada acrescenta um registro em `self.registros` com: usuário, pergunta mascarada, intenção, modelo, cache, bloqueado, latência, tokens e fontes. A resposta é um dict com `texto`, `fontes`, `latencia_s`, `tokens_in`, `tokens_out` e `modelo` (o mesmo formato do `nucleo`, para usar no `avaliar`).

_A célula grava `meu_assistente/assistente.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_assistente/assistente.py


### 10.6 **Rode o assistente.** Crie `assistente = Assistente(DOCUMENTOS, segredos=["NIM-SECRETO-42"])` e envie 8 mensagens variadas: perguntas normais, uma repetida (para o cache), uma paráfrase, um ataque, uma fora do escopo, um pedido de humano e uma com dados pessoais. Mostre as respostas e depois `assistente.registros` como tabela.

### 10.7 **Painel de monitoramento.** A partir de `registros`, calcule: total de mensagens, latência p50 e p90, % bloqueadas, % atendidas pelo cache, distribuição das intenções e tokens totais por modelo. Crie também um **alerta**: imprima AVISO se a p90 passar de 4 s ou se mais de 30% forem bloqueadas.

### 10.8 **Feedback explícito e implícito.** Simule eventos de feedback (já vem uma lista pronta para editar), com o polegar para cima/baixo e sinais implícitos (`reformulou` = o cliente perguntou de novo de outro jeito; `pediu_humano`). Junte com os registros pela posição e calcule a satisfação por intenção, contando os sinais implícitos como negativos.

### 10.9 **Feedback escondido na conversa.** Clientes raramente clicam em botões, mas escrevem coisas como "não era isso que eu perguntei". Para 4 mensagens de exemplo, peça ao modelo um JSON `{"sentimento": "positivo|negativo|neutro", "motivo": ...}` sobre a **resposta anterior** do assistente. Conte quantas indicam problema.

### 10.10 **A prova final.** Avalie o `assistente.responder` com o pipeline do capítulo 4 nas 8 perguntas e compare com a linha de base (`../saidas/base_cap04.csv`): nota do juiz, % com nota >= 4, latência e custo. Termine escrevendo, numa célula de texto, as 3 melhorias que você faria a seguir com base nos números.

## Para continuar evoluindo
- Rode o curso **sem o modo simulado** e compare os números com os do gabarito.
- Aumente o conjunto de avaliação com perguntas reais e mantenha a avaliação rodando a cada mudança de prompt ou de modelo.
- Exponha o assistente numa API (FastAPI) com *streaming*. O curso **RAG agêntico de produção**, nesta mesma pasta de estudos, faz isso com OpenSearch, Redis e LangGraph.
- Troque o `MODELO` por um modelo aberto (Llama, Qwen) via OpenRouter e veja o efeito na nota e no custo.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero testar com o modelo de verdade (fora do modo simulado):
